# Efficient Transformer

这一章展示一种更接近实际训练代码的通用 Transformer 语言模型实现.

和前面的 Decoder 教学版相比, 这里保留网络结构本身的可见性, 但把训练数据组织方式改成 packed sequence training: 一个 batch 内的多条样本拼接成一条长序列, 以减少 padding 浪费.

拼接之后必须解决两个问题:

1. 每个 token 仍然只能看见自己和过去 token, 也就是 causal attention.

2. 不同原始样本之间不能互相 attention, 否则模型会把两首诗错误地当成一条连续文本.

所以这里使用 variable-length attention mask: 同时表达因果约束和样本边界约束. attention 计算本身使用 PyTorch 内置的 `scaled_dot_product_attention`, tokenizer 使用 Hugging Face `AutoTokenizer`.

这里的 packing 由 `DataCollatorWithFlattening` 完成; tokenizer 内置的 `sep_token_id` 被复用为每首诗的结束 token.


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithFlattening


torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
class RoPE(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        inv_freq = base ** -(torch.arange(0, head_dim, 2).float() / head_dim)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, q, k, position_ids=None):
        if position_ids is None:
            seq_len = q.size(-2)
            position_ids = torch.arange(seq_len, device=q.device)[None, :]

        angles = position_ids[..., None].to(self.inv_freq.dtype) * self.inv_freq[None, None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[:, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[:, None]
        return q * cos + self.rotate_half(q) * sin, k * cos + self.rotate_half(k) * sin
    
    @staticmethod
    def rotate_half(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = RoPE(self.head_dim)
        self.dropout = dropout

    def forward(self, x, position_ids=None, seq_idx=None):
        batch_size, seq_len, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        q, k = self.rope(q, k, position_ids=position_ids)

        token_idx = torch.arange(seq_len, device=x.device)
        causal_mask = token_idx[:, None] >= token_idx[None, :]
        if seq_idx is None:
            attn_mask = causal_mask[None, None]
        else:
            same_sequence = seq_idx[:, :, None] == seq_idx[:, None, :]
            attn_mask = (same_sequence & causal_mask[None])[:, None]

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).reshape(batch_size, seq_len, d_model)
        return self.out(y)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w3(self.dropout(F.silu(self.w1(x)) * self.w2(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        self.norm1 = nn.RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, ffn_dim, dropout)

    def forward(self, x, position_ids=None, seq_idx=None):
        x = x + self.attn(self.norm1(x), position_ids=position_ids, seq_idx=seq_idx)
        x = x + self.ffn(self.norm2(x))
        return x


class EfficientTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=4, ffn_dim=768, dropout=0.1, padding_idx=None):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.RMSNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size, bias=False)
        self.out.weight = self.token_embedding.weight

    def forward(self, input_ids, position_ids=None, seq_idx=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x, position_ids=position_ids, seq_idx=seq_idx)
        return self.out(self.norm(x))

In [ ]:
batch_size = 64
subset_size = 80_000
max_length = 256
tokenizer_name = "uer/gpt2-chinese-cluecorpussmall"

dataset = load_dataset("larryvrh/Chinese-Poems")
subset = dataset["train"].select(range(min(subset_size, len(dataset["train"]))))

def format_poem(poem):
    text = f"{poem['title']} {poem['author']} {poem['dynasty']}\n{poem['content']}"
    return {"text": text}

text_dataset = subset.map(format_poem, desc="Formatting poems")
texts = text_dataset["text"]
char_lengths = [len(text) for text in texts]

print(f"样本数: {len(texts)}")
print(f"最大文本长度: {max(char_lengths)}")
print(f"平均文本长度: {sum(char_lengths) / len(char_lengths):.2f}")
print(f"超过 max_length={max_length} 的样本比例: {sum(length > max_length for length in char_lengths) / len(char_lengths):.2%}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

print("vocab_size:", tokenizer.vocab_size)
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("sep_token:", tokenizer.sep_token, tokenizer.sep_token_id)


def tokenize_fn(examples):
    outputs = tokenizer(
        examples["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=max_length - 1,
    )
    outputs["input_ids"] = [ids + [tokenizer.sep_token_id] for ids in outputs["input_ids"]]
    return outputs

tokenized_dataset = text_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=text_dataset.column_names,
    desc="Tokenizing"
)

token_lengths = [len(ids) for ids in tokenized_dataset["input_ids"]]
print(f"最大 token 长度: {max(token_lengths)}")
print(f"平均 token 长度: {sum(token_lengths) / len(token_lengths):.2f}")

In [ ]:
collate_fn = DataCollatorWithFlattening(
    return_position_ids=True,
    return_seq_idx=True,
    return_tensors="pt",
)

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=4,
)

print("train steps:", len(train_dataloader))

In [ ]:
model = EfficientTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    num_heads=8,
    num_layers=6,
    ffn_dim=768,
    dropout=0.1,
    padding_idx=tokenizer.pad_token_id,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
amp_context = torch.amp.autocast("cuda", enabled=use_amp)

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
epochs = 3

model.train()
for epoch in range(epochs):
    for step, batch in enumerate(train_dataloader, start=1):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        position_ids = batch["position_ids"].to(device)
        seq_idx = batch["seq_idx"].to(device)

        optimizer.zero_grad(set_to_none=True)
        with amp_context:
            logits = model(input_ids, position_ids=position_ids, seq_idx=seq_idx)
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                labels[:, 1:].contiguous().view(-1),
                ignore_index=-100,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if step % 10 == 0:
            ppl = torch.exp(loss.detach()).item()
            print(f"epoch {epoch+1}, step {step}, loss {loss.item():.4f}, ppl {ppl:.2f}")

## 生成与解码

`uer/gpt2-chinese-cluecorpussmall` 使用 WordPiece 风格 tokenizer, `decode` 会在中文 token 之间插入空格. 这里的 `decode_generated_text` 只删除中文字符之间的空格, 不直接删除所有空格. 需要注意, 这个 tokenizer 本身不会可靠保留输入里的空格和换行结构.


In [ ]:

@torch.no_grad()
def generate(prompt, max_new_tokens, temperature, top_k):
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)

    for _ in range(max_new_tokens):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, float("-inf"))
            next_logits = filtered.scatter(1, indices, values)

        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == tokenizer.sep_token_id:
            break
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


generate("静夜思 李白 唐代\n", max_new_tokens=256, temperature=0.5, top_k=30)